# CRM de PME, lido com calma

A Nexora é fictícia. A base sai de `data/generate.py` com a semente 42, de janeiro de 2023 a outubro de 2026. Este notebook não estima parâmetro nenhum: ele lê o DuckDB e aponta onde a conta engana.

Antes de rodar: `make generate` se o arquivo `data/warehouse/crm.duckdb` não existir.

In [1]:
from analysis.metrics import rollup_cohort, rollup_funnel, rollup_mrr
from analysis.queries import run_sql, run_sql_file

volumes = run_sql(
    """
    SELECT 'empresas' AS tabela, COUNT(*) AS linhas FROM companies
    UNION ALL SELECT 'leads', COUNT(*) FROM leads
    UNION ALL SELECT 'oportunidades', COUNT(*) FROM opportunities
    UNION ALL SELECT 'assinaturas', COUNT(*) FROM subscriptions
    UNION ALL SELECT 'ativas', COUNT(*) FROM subscriptions WHERE status = 'active'
    UNION ALL SELECT 'canceladas', COUNT(*) FROM subscriptions WHERE status = 'churned'
    """
)
volumes

,tabela,linhas
0,empresas,6763
1,leads,6763
2,oportunidades,1142
3,assinaturas,477
4,ativas,327
5,canceladas,150


## Funil

A taxa do período é a soma dos volumes. A média das taxas mensais é outra conta e, aqui, não coincide com o funil.

In [2]:
funil = rollup_funnel(run_sql_file("01_funnel_conversion.sql"))
funil

{'leads': 6763.0,
 'mqls': 2644.0,
 'sqls': 1475.0,
 'opportunities': 1142.0,
 'wins': 477.0,
 'lead_to_mql': 0.39095076149637736,
 'mql_to_sql': 0.5578668683812406,
 'sql_to_opportunity': 0.7742372881355932,
 'opportunity_to_win': 0.4176882661996497,
 'lead_to_win': 0.0705308295135295}

## Receita

NRR e GRR usam o MRR do início do mês. Cliente novo fica de fora. ARR de run-rate é o MRR do mês vezes 12, não a soma do ano.

In [3]:
receita = rollup_mrr(run_sql_file("03_mrr_arr_nrr.sql"))
receita.tail(3)[
    ["month_start", "ending_customers", "ending_mrr", "ending_arr", "nrr", "grr", "logo_churn_rate"]
]

,month_start,ending_customers,ending_mrr,ending_arr,nrr,grr,logo_churn_rate
40,2026-08-01,316,414610.15,4975321.80,0.993774,0.988114,0.019293
41,2026-09-01,326,417138.86,5005666.32,0.968671,0.962350,0.028481
42,2026-10-01,327,425289.56,5103474.72,0.993539,0.986675,0.024540


## Coorte

Idade 0 tem retenção 1. Daí para a frente, receita e logo se separam quando há expansão. Coorte nova ainda não chegou na idade longa: o ponto de 18 meses não representa quem entrou neste ano.

In [4]:
coorte = rollup_cohort(run_sql_file("04_cohort_retention.sql"))
idades = coorte[coorte["age_months"].isin([0, 6, 12, 18])]
resumo = idades.groupby("age_months", as_index=False)[
    ["cohort_customers", "active_customers", "cohort_initial_mrr", "retained_mrr"]
].sum()
resumo["logos"] = resumo["active_customers"] / resumo["cohort_customers"]
resumo["receita"] = resumo["retained_mrr"] / resumo["cohort_initial_mrr"]
resumo[["age_months", "cohort_customers", "logos", "receita"]]

,age_months,cohort_customers,logos,receita
0,0,477,1.000000,1.000000
1,6,394,0.883249,0.881157
2,12,338,0.748521,0.790299
3,18,249,0.702811,0.773550


## Propensão a cancelar

O relatório abaixo foi gravado por `python -m analysis.churn_model`. O corte do tempo está no módulo: treino até setembro de 2025, teste em março e junho de 2026, rótulo de 90 dias.

O gerador faz parte de quem vai cancelar abrir mais chamado e atrasar fatura. O AUC mede esse mecanismo. Não é um CRM real. Empresas que já estavam no treino puxam o AUC para cima; o número entre empresas inéditas é o que eu levaria para uma conversa.

In [5]:
import json
from pathlib import Path

relatorio = json.loads(Path("outputs/churn_model.json").read_text(encoding="utf-8"))
chaves = [
    "logistic_roc_auc",
    "logistic_roc_auc_ci95",
    "gradient_boosting_roc_auc",
    "rule_roc_auc",
    "unseen_company_auc",
    "test_rows",
    "test_positives",
    "test_precision",
    "test_recall",
    "logistic_brier",
    "baseline_brier",
]
{chave: relatorio[chave] for chave in chaves}

{'logistic_roc_auc': 0.7726733576642337,
 'logistic_roc_auc_ci95': [0.6815270095148842, 0.8633297080931466],
 'gradient_boosting_roc_auc': 0.7630474452554744,
 'rule_roc_auc': 0.6811587591240875,
 'unseen_company_auc': 0.6796218487394958,
 'test_rows': 588,
 'test_positives': 40,
 'test_precision': 0.5,
 'test_recall': 0.475,
 'logistic_brier': 0.045762535726307584,
 'baseline_brier': 0.06348280588516209}